# 基本介绍

LangChain 1.2.x 的长期记忆基于 store 持久化数据，相关的API有：
- `put()`：负责写入
- `get()`：负责读取
- `search()`：负责检索

我们可以在Agent执行流程之外直接访问长期记忆。

## ① put()源码剖析
直接调用`put()`函数即可，函数签名如下
```python
def put(
    self,
    namespace: tuple[str, ...],
    key: str,
    value: dict[str, Any],
    index: Literal[False] | list[str] | None = None,
    *,
    ttl: float | None | NotProvided = NOT_PROVIDED,
) -> None:
```

**参数说明**
- `namespace`: 文档所在的层级路径
- `key`: 该路径下的唯一键
- `value`: 要保存的JSON‑like字典
- `index`: 控制语义检索索引
  - `None`(默认选项): 使用 store 初始化时配置的索引配置，如果初始化时没有指定索引策略，则 index参数将会被忽略
  - `False`: 不为该 item 建立语义索引
  - `list[str]`: 只对指定字段路径建索引
- `ttl`: 可选，过期时间；是否支持取决于具体 store 实现

**举例：**
```python
store.put(
    ("users", "alice", "memories"), # namespace
    "pref_food",                    # key
    {"category": "food", "text": "Alice likes sushi"} # value
)
```

## ② get()源码剖析
按照 `namespace + key` 精确查询，返回的不止是 `value`，而是完整对象。即LangGraph底层将数据封装为 `Item` 对象。

函数签名如下
```python
def get(
    self,
    namespace: tuple[str, ...],
    key: str,
    *,
    refresh_ttl: bool | None = None,
) -> Item | None:
```

**参数说明**
- `namespace`: 文档所在的层级路径
- `key`: 该路径下的唯一键
- `refresh_ttl`: 是否刷新当前item的ttl (time‑to‑live，存活时间)
  - 默认为`None`：表示采用创建store对象时指定的同名配置
  - 如果没有配置TTL，该参数被忽略。

**举例：**
```python
item = my_store.get(("users", "alice", "memories"), "pref_food")
if item is not None:
    print(item.value)
    # {'category': 'food', 'text': 'Alice likes sushi'}
```

# InMemory示例


In [1]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()
namespace = ("users",)
user_id = 'user-1'
username = "小蓝"
store.put(namespace, user_id, {"name": username})
print(store.get(namespace, user_id))

Item(namespace=['users'], key='user-1', value={'name': '小蓝'}, created_at='2026-08-31T07:45:46.028670+00:00', updated_at='2026-08-31T07:45:46.028672+00:00')


In [2]:

store.put(namespace, user_id, {"name": '小红'})
print(store.get(namespace, user_id))

Item(namespace=['users'], key='user-1', value={'name': '小红'}, created_at='2026-08-31T07:45:49.153504+00:00', updated_at='2026-08-31T07:45:49.153506+00:00')


# PostgresStore示例

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.store.postgres import PostgresStore
import os
from dotenv import load_dotenv
load_dotenv(override=True)

DB_URL = os.getenv("DATABASE_URL")
with PostgresStore.from_conn_string(DB_URL) as store:
    # 初始化PostgreSQL数据库
    store.setup()
    namespace = ("users",)
    user_id = 'user-1'
    username = "小公园"
    store.put(namespace, user_id, {"name": username})
    print(store.get(namespace, user_id))

Item(namespace=['users'], key='user-1', value={'name': '小公园'}, created_at='2026-08-31T08:04:14.934685+00:00', updated_at='2026-08-31T08:04:14.934685+00:00')


In [4]:
with PostgresStore.from_conn_string(DB_URL) as store:
    # 初始化PostgreSQL数据库
    store.setup()
    namespace = ("users",)
    user_id = 'user-1'
    username = "小公园-2"
    store.put(namespace, user_id, {"name": username})
    print(store.get(namespace, user_id))

Item(namespace=['users'], key='user-1', value={'name': '小公园-2'}, created_at='2026-08-31T08:04:14.934685+00:00', updated_at='2026-08-31T08:04:17.403199+00:00')
